In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install transformers datasets seqeval peft accelerate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [4]:
from google.colab import files
uploaded = files.upload()

Saving annotations.json to annotations.json


In [5]:
import json

with open("annotations.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Total queries loaded: {len(data)}")
print(f"First query: {data[0]['query']}")
print(f"Intent: {data[0]['intent']}")
print(f"Entities: {data[0]['entities']}")

Total queries loaded: 1000
First query: Nike wala red kurti size M teen hazaar ke andar chahiye fast delivery
Intent: SEARCH
Entities: [{'text': 'Nike', 'type': 'BRAND'}, {'text': 'red', 'type': 'ATTRIBUTE'}, {'text': 'kurti', 'type': 'PRODUCT'}, {'text': 'size M', 'type': 'SIZE'}, {'text': 'teen hazaar ke andar', 'type': 'PRICE_RANGE'}, {'text': 'fast delivery', 'type': 'DELIVERY_CONSTRAINT'}]


In [6]:
# Intent labels → numbers
INTENT2ID = {
    "SEARCH": 0,
    "COMPARE": 1,
    "BUY": 2,
    "TRACK": 3,
    "RETURN": 4
}
ID2INTENT = {v: k for k, v in INTENT2ID.items()}

# NER labels → numbers (BIO format)
# B = Beginning of entity, I = Inside entity, O = Outside
NER2ID = {
    "O": 0,
    "B-PRODUCT": 1,        "I-PRODUCT": 2,
    "B-BRAND": 3,          "I-BRAND": 4,
    "B-ATTRIBUTE": 5,      "I-ATTRIBUTE": 6,
    "B-PRICE_RANGE": 7,    "I-PRICE_RANGE": 8,
    "B-SIZE": 9,           "I-SIZE": 10,
    "B-DELIVERY_CONSTRAINT": 11, "I-DELIVERY_CONSTRAINT": 12
}
ID2NER = {v: k for k, v in NER2ID.items()}

print(f"Intent classes: {len(INTENT2ID)}")
print(f"NER labels: {len(NER2ID)}")
print(f"Example: SEARCH → {INTENT2ID['SEARCH']}")
print(f"Example: B-PRICE_RANGE → {NER2ID['B-PRICE_RANGE']}")

Intent classes: 5
NER labels: 13
Example: SEARCH → 0
Example: B-PRICE_RANGE → 7


In [7]:
def create_bio_labels(query, entities):
    words = query.split()
    labels = ["O"] * len(words)

    for entity in entities:
        entity_text = entity["text"]
        entity_type = entity["type"]
        entity_words = entity_text.split()
        entity_len = len(entity_words)

        # Find where this entity appears in the query words
        for i in range(len(words) - entity_len + 1):
            if words[i:i+entity_len] == entity_words:
                labels[i] = f"B-{entity_type}"
                for j in range(1, entity_len):
                    labels[i+j] = f"I-{entity_type}"
                break

    return words, labels

# Test on first query
words, labels = create_bio_labels(data[0]["query"], data[0]["entities"])
for word, label in zip(words, labels):
    print(f"{word:25} → {label}")

Nike                      → B-BRAND
wala                      → O
red                       → B-ATTRIBUTE
kurti                     → B-PRODUCT
size                      → B-SIZE
M                         → I-SIZE
teen                      → B-PRICE_RANGE
hazaar                    → I-PRICE_RANGE
ke                        → I-PRICE_RANGE
andar                     → I-PRICE_RANGE
chahiye                   → O
fast                      → B-DELIVERY_CONSTRAINT
delivery                  → I-DELIVERY_CONSTRAINT


In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/muril-base-cased")
print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Testing tokenization:")
test = "teen hazaar ke andar red kurti chahiye"
tokens = tokenizer.tokenize(test)
print(tokens)

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

Vocabulary size: 197258
Testing tokenization:
['teen', 'hazaar', 'ke', 'andar', 'red', 'ku', '##rti', 'chahiye']


In [9]:
def tokenize_and_align(query, entities, max_length=128):
    words, word_labels = create_bio_labels(query, entities)

    encoding = tokenizer(
        words,
        is_split_into_words=True,
        max_length=max_length,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    word_ids = encoding.word_ids(batch_index=0)

    aligned_labels = []
    previous_word_id = None

    for word_id in word_ids:
        if word_id is None:
            # Special tokens [CLS] and [SEP] — ignore
            aligned_labels.append(-100)
        elif word_id != previous_word_id:
            # First token of a new word — use real label
            aligned_labels.append(NER2ID[word_labels[word_id]])
        else:
            # Subsequent subword tokens — ignore
            aligned_labels.append(-100)
        previous_word_id = word_id

    return {
        "input_ids": encoding["input_ids"].squeeze(),
        "attention_mask": encoding["attention_mask"].squeeze(),
        "intent_label": INTENT2ID[query_data["intent"]],
        "ner_labels": aligned_labels
    }

# Test on first query
query_data = data[0]
result = tokenize_and_align(
    query_data["query"],
    query_data["entities"]
)

print(f"Input IDs shape: {result['input_ids'].shape}")
print(f"Attention mask shape: {result['attention_mask'].shape}")
print(f"Intent label: {result['intent_label']} ({ID2INTENT[result['intent_label']]})")
print(f"NER labels (first 20): {result['ner_labels'][:20]}")
print(f"Non -100 NER labels: {[l for l in result['ner_labels'] if l != -100]}")

Input IDs shape: torch.Size([128])
Attention mask shape: torch.Size([128])
Intent label: 0 (SEARCH)
NER labels (first 20): [-100, 3, 0, -100, 5, 1, -100, 9, 10, 7, 8, 8, 8, 0, 11, 12, -100, -100, -100, -100]
Non -100 NER labels: [3, 0, 5, 1, 9, 10, 7, 8, 8, 8, 0, 11, 12]


In [11]:
import torch
from torch.utils.data import Dataset

class VaakSetuDataset(Dataset):
    def __init__(self, data):
        self.samples = []
        skipped = 0

        for item in data:
            try:
                words, word_labels = create_bio_labels(
                    item["query"],
                    item["entities"]
                )

                encoding = tokenizer(
                    words,
                    is_split_into_words=True,
                    max_length=128,
                    padding="max_length",
                    truncation=True,
                    return_tensors="pt"
                )

                word_ids = encoding.word_ids(batch_index=0)
                aligned_labels = []
                previous_word_id = None

                for word_id in word_ids:
                    if word_id is None:
                        aligned_labels.append(-100)
                    elif word_id != previous_word_id:
                        aligned_labels.append(NER2ID[word_labels[word_id]])
                    else:
                        aligned_labels.append(-100)
                    previous_word_id = word_id

                self.samples.append({
                    "input_ids": encoding["input_ids"].squeeze(),
                    "attention_mask": encoding["attention_mask"].squeeze(),
                    "intent_label": torch.tensor(INTENT2ID[item["intent"]]),
                    "ner_labels": torch.tensor(aligned_labels)
                })

            except Exception as e:
                skipped += 1

        print(f"Processed: {len(self.samples)} queries")
        print(f"Skipped: {skipped} queries")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

dataset = VaakSetuDataset(data)

Processed: 1000 queries
Skipped: 0 queries


In [12]:
from torch.utils.data import random_split

# 80% train, 10% validation, 10% test
train_size = 800
val_size = 100
test_size = 100

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"Train: {len(train_dataset)} queries")
print(f"Validation: {len(val_dataset)} queries")
print(f"Test: {len(test_dataset)} queries")

Train: 800 queries
Validation: 100 queries
Test: 100 queries


In [13]:
import os
import pickle

# Create VaakSetu folder in Drive
save_dir = "/content/drive/MyDrive/VaakSetu"
os.makedirs(save_dir, exist_ok=True)

# Save datasets
torch.save(train_dataset, f"{save_dir}/train_dataset.pt")
torch.save(val_dataset, f"{save_dir}/val_dataset.pt")
torch.save(test_dataset, f"{save_dir}/test_dataset.pt")

# Save label mappings
with open(f"{save_dir}/label_mappings.json", "w") as f:
    json.dump({
        "INTENT2ID": INTENT2ID,
        "ID2INTENT": ID2INTENT,
        "NER2ID": NER2ID,
        "ID2NER": ID2NER
    }, f)

print("Saved to Google Drive:")
print(f"  {save_dir}/train_dataset.pt")
print(f"  {save_dir}/val_dataset.pt")
print(f"  {save_dir}/test_dataset.pt")
print(f"  {save_dir}/label_mappings.json")

Saved to Google Drive:
  /content/drive/MyDrive/VaakSetu/train_dataset.pt
  /content/drive/MyDrive/VaakSetu/val_dataset.pt
  /content/drive/MyDrive/VaakSetu/test_dataset.pt
  /content/drive/MyDrive/VaakSetu/label_mappings.json
